# 01_data_prep.ipynb

Starter notebook: load CSVs, quick checks, and RFM example.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

DATA_DIR = '../data'  # adjust if running from repo root


In [ ]:
customers = pd.read_csv('../data/customers.csv', parse_dates=['signup_date'])
products = pd.read_csv('../data/products.csv')
orders = pd.read_csv('../data/orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv('../data/order_items.csv')
campaigns = pd.read_csv('../data/campaigns.csv')

print('Loaded tables: customers, products, orders, order_items, campaigns')


In [ ]:
display(customers.head())
display(orders.head())


In [ ]:
# Basic cleaning examples
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders = orders[orders['order_amount'] >= 0]
# Remove duplicates if any
orders = orders.drop_duplicates(subset=['order_id'])
order_items = order_items.drop_duplicates(subset=['order_item_id'])


In [ ]:
# RFM example (snapshot = max order_date + 1 day)
snapshot = orders['order_date'].max() + pd.Timedelta(days=1)
print('Snapshot:', snapshot)

# Aggregate
agg = orders[orders['order_status']=='completed'].groupby('customer_id').agg({
    'order_date': lambda x: (snapshot - x.max()).days,
    'order_id': 'nunique',
    'order_amount': 'sum'
}).reset_index()
agg.columns = ['customer_id','recency_days','frequency','monetary']
# Basic RFM bins
agg['r_bin'] = pd.qcut(agg['recency_days'].rank(method='first'), 5, labels=[5,4,3,2,1])
agg['f_bin'] = pd.qcut(agg['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
agg['m_bin'] = pd.qcut(agg['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5])
agg['rfm_score'] = agg['r_bin'].astype(str) + agg['f_bin'].astype(str) + agg['m_bin'].astype(str)

agg.head()


In [ ]:
# Save processed sample
agg.to_csv('../data/processed_rfm.csv', index=False)
print('Saved processed_rfm.csv')
